In [ ]:
import pathlib
import pandas as pd
import seaborn as sns
import yaml
import warnings
import scanpy as sc

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

plot_dir = pathlib.Path("plots/figure3")
plot_dir.mkdir(exist_ok=True)

_IN_STUDY = {
    "kim_lung": ("kim_3",),
    "neftel_smartseq2_brain": ("neftel_4", "neftel_6"),
    "neftel_10x_brain": ("neftel_4", "neftel_6"),
    "wu_breast": ("wu_4", "wu_6"),
    "ji_skin": ("ji_3",),
    # "vazquez_ovarian": ("Vazquez_6",)
}


font_config = {
    "axes.titlesize": 20,  # Title font size
    "axes.labelsize": 16,  # X and Y label font size
    "legend.title_fontsize": 16,  # Legend title font size
    "legend.fontsize": 16,  # Legend text font size
    "xtick.labelsize": 14,  # X-axis tick labels font size
    "ytick.labelsize": 14,  # Y-axis tick labels font size
}

_IN_STUDY_ORDER = (
    "Ji (3)",
    "Neftel, SS2 (4)",
    "Neftel, SS2 (6)",
    "Neftel, 10X (4)",
    "Neftel, 10X (6)",
    "Kim (3)",
    "Wu (4)",
    "Wu (6)",
)

In [ ]:
figure_config = yaml.safe_load(open("figures_config.yml", "r"))
dataset_map = figure_config["datasets"]
method_order = figure_config["method_order"]
method_colormap = figure_config["method_colormap"]
methods_map = figure_config["methods"]
scenarios = {}

for config_path in pathlib.Path("../configs/signatures").iterdir():
    if config_path.stem in ["config", "debug"]:
        continue
    config = yaml.safe_load(open(config_path, "r"))
    for scenario, values in config["scenarios"].items():
        values["root"] = config["root"]
        if "gbm_subsample" in config["root"]:
            continue
        if "sparse" in config["root"]:
            continue
        scenarios[scenario] = values

results = []
study_path = pathlib.Path("../results/signatures/gbm_subsample/scores/method/scrna")

for method_path in study_path.iterdir():
    method = method_path.stem
    for scenario_path in method_path.iterdir():
        signature_dir = scenario_path / "4000" / "True"
        for signature_path in signature_dir.iterdir():
            df = pd.read_csv(signature_path, index_col=0)
            df["Method"] = methods_map[method]
            df["Signature"] = signature_path.stem
            dataset, n_sample, random_seed = scenario_path.stem.rsplit("_", maxsplit=2)
            df["Dataset"] = dataset
            df["Random_seed"] = random_seed
            df["n_patients"] = int(n_sample)
            results.append(df)
results = pd.concat(results)
results = results[results["Signature"] == "neftel_4"]

results = results.set_index(["n_patients", "Random_seed"])

In [ ]:
results = results[results["Method"] == "MNC"]

In [ ]:
results

In [ ]:
results["sample"] = None

for path in pathlib.Path(
    "../results/signatures/gbm_subsample/data/preprocessed"
).iterdir():
    if path.name.endswith(".rds"):
        continue
    if path.name == "neftel_smartseq2_brain.h5ad":
        continue
    dataset, n_sample, random_seed = path.stem.rsplit("_", maxsplit=2)
    obs = sc.read_h5ad(path).obs
    bad_sample = obs["sample"].unique().isin(["BT830", "BT1160"]).any()
    results.loc[(int(n_sample), str(random_seed)), ["sample"]] = bad_sample

In [ ]:
results.reset_index()

In [ ]:
sns.boxenplot(results.reset_index(), x="sample", y="0")

In [ ]:
path.name.rsplit("_", maxsplit=2)